In [0]:
import urllib.request
import json

def testar_conectividade(nome, url):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "test"})
        with urllib.request.urlopen(req, timeout=10) as resp:
            print(f"[OK] {nome}: status {resp.status}")
            return True
    except Exception as e:
        print(f"[FALHOU] {nome}: {type(e).__name__} - {e}")
        return False

# Teste 1: brapi.dev (fonte de dados)
testar_conectividade("brapi.dev", "https://brapi.dev/api/quote/PETR4")

# Teste 2: GitHub API (para o agente de commit, mais adiante)
testar_conectividade("GitHub API", "https://api.github.com")

# Teste 3: github.com direto
testar_conectividade("GitHub", "https://github.com")

In [0]:
# preview - registros de teste candidatos a remocao
display(spark.sql("""
    SELECT execucao_id, notebook, data_referencia, status, mensagem_erro, inicio, duracao_segundos
    FROM poc_b3_modernizacao.observability.pipeline_runs
    WHERE mensagem_erro LIKE '%quote-invalida%'
       OR mensagem_erro LIKE '%caminho_invalido%'
       OR mensagem_erro LIKE '%tabela_invalida%'
       OR (notebook = '01_ingestao_landing' AND inicio = '2026-09-01T17:50:50.353+00:00')
    ORDER BY inicio
"""))

In [0]:
# limpeza - remove registros de teste (falhas proposital + sucessos confusos do mesmo lote de teste)
resultado = spark.sql("""
    DELETE FROM poc_b3_modernizacao.observability.pipeline_runs
    WHERE mensagem_erro LIKE '%quote-invalida%'
       OR mensagem_erro LIKE '%caminho_invalido%'
       OR mensagem_erro LIKE '%tabela_invalida%'
       OR (notebook = '01_ingestao_landing' AND inicio = '2026-09-01T17:50:50.353+00:00')
""")
print("Limpeza concluida.")

In [0]:
# valida limpeza
display(spark.table("poc_b3_modernizacao.observability.pipeline_runs").orderBy("inicio"))

In [0]:
# limpeza - remove os 3 registros confusos de sucesso (mesmo lote de teste, inicio nao atualizado entre execucoes)
resultado = spark.sql("""
    DELETE FROM poc_b3_modernizacao.observability.pipeline_runs
    WHERE execucao_id IN (
        '81e696a6-937b-44f7-80f3-4c1209d16d60',
        '839f37ba-d988-4521-9d57-deb9f2959b04',
        '172229e7-dfd6-47c3-89bf-ef42e116d968'
    )
""")
print("Limpeza concluida.")

In [0]:
display(spark.table("poc_b3_modernizacao.observability.pipeline_runs").orderBy("inicio"))

In [0]:
%sql
SELECT inicio, notebook, tipo_anomalia, detalhe, causa_raiz
FROM poc_b3_modernizacao.observability.auditoria_anomalias
WHERE causa_raiz IS NULL
ORDER BY inicio DESC

In [0]:
spark.sql("""
    UPDATE poc_b3_modernizacao.observability.auditoria_anomalias
    SET causa_raiz = 'Execucao manual, celula por celula, com pausas, durante testes da correcao do Widget modo_execucao (04/09) - duracao inflada pelo processo de teste, nao reflete lentidao real do notebook.'
    WHERE notebook = '03_silver'
      AND inicio >= to_timestamp('2026-09-04T14:22:00') AND inicio <= to_timestamp('2026-09-04T14:23:00')
""")

spark.sql("""
    UPDATE poc_b3_modernizacao.observability.auditoria_anomalias
    SET causa_raiz = 'Multiplas execucoes manuais proximas durante testes da correcao do Widget modo_execucao em varios notebooks do pipeline (04/09) - nao e anomalia de producao.'
    WHERE notebook = '05_reconciliacao'
      AND inicio >= to_timestamp('2026-09-04T16:32:00') AND inicio <= to_timestamp('2026-09-04T16:33:00')
""")

print("Causas raiz corrigidas (metodo por intervalo).")

In [0]:
spark.sql("""
    UPDATE poc_b3_modernizacao.observability.auditoria_anomalias
    SET causa_raiz = 'Primeira execucao deste notebook (02/09) - criacao inicial das tabelas auditoria_anomalias e auditoria_gaps (CREATE TABLE), mais lento que as execucoes seguintes (INSERT via inserir_se_novo em tabela ja existente).'
    WHERE notebook = '07_auditoria_execucoes'
      AND inicio >= to_timestamp('2026-09-02T18:48:00') AND inicio <= to_timestamp('2026-09-02T18:49:00')
""")
print("Causa raiz preenchida.")